# MobileNetV2 Baseline Training — CIFAR-10

This notebook trains the FP32 MobileNetV2 baseline used for the quantization experiments. The training history is recorded during the run and used to generate the accuracy and loss curves; no metric values are hardcoded.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Model and dataset setup


In [ ]:
mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2435, 0.2616]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))


In [ ]:
model = models.mobilenet_v2(num_classes=10).to(device)
print(model)


## 2. Training configuration


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
epochs = 20
checkpoint_path = "checkpoints/mobilenetv2_cifar10_fp32.pth"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

# Metrics are collected from the actual run.
train_losses = []
train_accuracies = []
test_accuracies = []
learning_rates = []
best_test_accuracy = 0.0


## 3. Train the baseline model


In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    train_accuracy = 100.0 * correct / total

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)
            test_total += labels.size(0)
            test_correct += (predictions == labels).sum().item()

    test_accuracy = 100.0 * test_correct / test_total
    current_lr = optimizer.param_groups[0]["lr"]

    train_losses.append(epoch_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
    learning_rates.append(current_lr)

    scheduler.step()

    print(
        f"Epoch [{epoch + 1}/{epochs}] | "
        f"Train Loss: {epoch_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Test Accuracy: {test_accuracy:.2f}% | "
        f"Learning Rate: {current_lr:.6f}"
    )

    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Best model saved! Test Accuracy: {best_test_accuracy:.2f}%")


## 4. Accuracy and loss curves

Both plots use the epoch-level metrics collected above from this training run. No accuracy or loss values are hardcoded.

In [ ]:
import matplotlib.pyplot as plt

plot_epochs = range(1, len(train_accuracies) + 1)

plt.figure(figsize=(8, 5))
plt.plot(plot_epochs, train_accuracies, marker="o", label="Train accuracy")
plt.plot(plot_epochs, test_accuracies, marker="o", label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("MobileNetV2 CIFAR-10 Accuracy")
plt.xticks(plot_epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(plot_epochs, train_losses, marker="o", label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("MobileNetV2 CIFAR-10 Training Loss")
plt.xticks(plot_epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

os.makedirs("figures", exist_ok=True)
plt.figure(figsize=(8, 5))
plt.plot(plot_epochs, train_accuracies, marker="o", label="Train accuracy")
plt.plot(plot_epochs, test_accuracies, marker="o", label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("MobileNetV2 CIFAR-10 Accuracy")
plt.xticks(plot_epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("figures/training_test_accuracy.png", dpi=160)
plt.show()


## 5. Save and verify the best FP32 checkpoint


In [ ]:
best_model = models.mobilenet_v2(num_classes=10).to(device)
best_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
best_model.eval()

correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = best_model(images)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

fp32_accuracy = 100.0 * correct / total
checkpoint_size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
num_parameters = sum(p.numel() for p in best_model.parameters())

print(f"Best recorded test accuracy: {best_test_accuracy:.2f}%")
print(f"FP32 test accuracy after reload: {fp32_accuracy:.2f}%")
print(f"Number of parameters: {num_parameters:,}")
print(f"Checkpoint size: {checkpoint_size_mb:.2f} MB")
print(f"Checkpoint path: {checkpoint_path}")
